# Train YOLOv8 Gun Detector (Kaggle kernel ready)

This notebook is adapted for Kaggle notebooks. It expects the dataset to be added as a Kaggle Dataset (see instructions below) and will train a YOLOv8 model using the dataset mounted at /kaggle/input/<dataset-slug>. Follow cells in order.

## Overview and prerequisites
- Create a Kaggle Dataset containing your folder `Gun with webcam views.v1i.yolov8` (upload as a dataset) or upload images/labels as a dataset on kaggle.com.
- Once the dataset is created, add it to your kernel via the "Add Data" button and select the dataset. The dataset will be available at `/kaggle/input/<dataset-slug>`.
- This notebook installs ultralytics and runs training. Kaggle provides GPUs (if selected) — choose a GPU accelerator in the kernel settings.


## 1) Install dependencies
Install Ultralytics (YOLOv8) and YAML helper. Kaggle kernels allow pip installs in the session.

In [ ]:
# Install ultralytics (choose a stable compatible version)
!pip install -q ultralytics==8.4.116 pyyaml
# Verify yolo CLI is available after install
!which yolo || true

## 2) Check GPU availability
Verify PyTorch/CUDA is visible in this kernel.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        print('device name:', torch.cuda.get_device_name(0))
    except Exception as e:
        print('cuda device name error:', e)


## 3) Point to the Kaggle dataset path
After you add your dataset via the Kaggle UI, set DATASET_SLUG below to the dataset folder name under /kaggle/input.

In [ ]:
# Set this to the dataset folder name shown under /kaggle/input after adding the dataset to the kernel
DATASET_SLUG = 'gun-with-webcam-views-v1i-yolov8'  # replace with your actual dataset folder name under /kaggle/input
DATASET_PATH = f'/kaggle/input/{DATASET_SLUG}'
print('Dataset path:', DATASET_PATH)
# Removed rsync duplication code to prevent slow Kaggle uploads

## 4) Inspect dataset and data.yaml
Show the tree and validate data.yaml contents. The notebook expects /kaggle/input/<slug>/data.yaml.

In [ ]:
!ls -la /kaggle/input/{DATASET_SLUG} || true
# show data.yaml if present
import yaml, os
cfg_path = f'/kaggle/input/{DATASET_SLUG}/data.yaml'
cfg = None
if os.path.exists(cfg_path):
    print('Found data.yaml at', cfg_path)
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    print(cfg)
if cfg is None:
    print('data.yaml not found. Please ensure data.yaml exists in the dataset root and contains nc and names fields.')
else:
    if 'nc' in cfg and 'names' in cfg:
        if cfg['nc'] != len(cfg['names']):
            print('WARNING: nc != len(names)')
        print('Classes:', cfg['names'])
    else:
        print('data.yaml missing nc/names; verify format')


## 5) Optional: Merge person dataset (COCO) with gun dataset
If you want a combined person+gun model, create a merged dataset where person=0 and gun=1. This is optional and requires careful label remapping. Steps are provided but commented out for safety.

In [ ]:
# OPTIONAL: merging snippet (commented)
# High level: copy COCO-person images/labels to merged folder, copy gun images/labels and remap gun class id->1
# Use this carefully and verify labels after remapping.
# Example remap code (DO NOT RUN without review):
# import glob, os
# gun_labels = glob.glob('/kaggle/working/dataset/train/labels/*.txt')
# for lf in gun_labels:
#     lines = open(lf).read().splitlines()
#     new = []
#     for ln in lines:
#         parts = ln.split()
#         parts[0] = '1'
#         new.append(' '.join(parts))
#     open(lf,'w').write('
'.join(new))
# After remap, create merged data.yaml with names: ['person','gun'] and correct paths.


## 6) Training cells
Run a quick test first (epochs=5, imgsz=320) then run the full training. Set name to a unique run folder. On Kaggle, default workers and batch sizes should be tuned for the available GPU and RAM.

In [ ]:
# Quick test (validate dataset and pipeline using data.yaml in the input folder)
!yolo task=detect mode=train model=yolov8n.pt data=/kaggle/input/gun-with-webcam-views-v1i-yolov8/data.yaml epochs=5 imgsz=320 batch=8 device=0 name=kaggle_quick_test || true
# Full run example (uncomment to run). Reduce batch if OOM.
# !yolo task=detect mode=train model=yolov8n.pt data=/kaggle/input/gun-with-webcam-views-v1i-yolov8/data.yaml epochs=50 imgsz=640 batch=16 device=0 name=kaggle_full_run


## 7) After training: locate best.pt and move to /kaggle/working for download
Kaggle stores outputs under /kaggle/working/runs/... or /kaggle/working. Copy best.pt to /kaggle/working so it appears in the Output files panel for download.

In [ ]:
# Example: copy best.pt from runs/detect/<name>/weights to /kaggle/working model folder
RUN_NAME = 'kaggle_full_run'  # set after your run name
src = f'/kaggle/working/runs/detect/{RUN_NAME}/weights/best.pt'
dst = '/kaggle/working/gun_best.pt'
import os
if os.path.exists(src):
    !cp {src} {dst}
    print('Copied best.pt to', dst)
else:
    print('best.pt not found at', src, ' - check runs/detect/<name>/weights')


## 8) Inference test
Run inference on a sample image and write a result image that you can download from the Output panel.

In [ ]:
from ultralytics import YOLO
import glob, os
imgs = glob.glob('/kaggle/input/gun-with-webcam-views-v1i-yolov8/**/images/*.*', recursive=True)
if imgs:
    img0 = imgs[0]
    print('Running inference on', img0)
    weights = '/kaggle/working/gun_best.pt'
    model_path = weights if os.path.exists(weights) else 'yolov8n.pt'
    model = YOLO(model_path)
    res = model(img0)
    out = '/kaggle/working/pred.jpg'
    try:
        res[0].plot(save=True, fname=out)
        print('Saved prediction to', out)
    except Exception as e:
        print('Inference/plot error:', e)
else:
    print('No images found in dataset images folders')


## 9) Update your project after downloading best.pt
After downloading gun_best.pt from the kernel Output, copy it into your project (e.g. app/models/gun_best.pt) and update detection.py as shown below.

In [ ]:
print('1) Place gun_best.pt into app/models/gun_best.pt in your project')
print("2) In app/detection.py change model path and flags:\n   model_path = 'app/models/gun_best.pt'\n   WEAPON_CLASS_IDS = [0]\n   SIMULATE_SECOND_PERSON_AS_WEAPON = False")


## 10) Troubleshooting / tips
- OOM: reduce batch or imgsz, or switch to yolov8n/yolov8s.
- Use smaller imgsz during prototyping and larger for final training.
- Kaggle kernels have time limits and GPU availability can vary; use kernel commit to save outputs.
- Verify label format: YOLO txt files: class x_center y_center width height (normalized).
